### Phase 2 — Data Validation and Cleaning

Read raw datasets, clean them, and save the results separately.


In [19]:
import pandas as pd
import json
from pathlib import Path
from pandas.api.types import is_numeric_dtype

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


### <b>Phase 2 - Data Validation and Cleaning</b>
- <b>2.1 Profile:</b> Inspect row/column counts, inferred data types, sample records, missing values, and ranges.
- <b>2.2 Missing values:</b> Choose drop, fill, or retain, explaining why; do not indiscriminately delete rows.
- <b>2.3 Duplicates:</b> Find duplicates using stable identifiers (product URL / product ID / date), not only entire-row equality.
- <b>2.4 Text and numbers:</b> Convert book prices and rating strings into meaningful numeric fields; handle currency symbols and separators.
- <b>2.5 Schemas:</b> Use consistent snake_case headers and validate price, rating, date, and stock OHLCV types.
- <b>2.6 Export:</b> Write cleaned product and book CSVs; export at least one cleaned dataset to XML as a file-format exercise.
- <b>2.7 Quality log:</b> Record counts before/after cleaning, transformations, excluded rows, and unresolved concerns.
- Suggested cleaned outputs: data/clean/books_clean.csv, data/clean/products_clean.csv, data/clean/amzn_clean.csv, data/clean/products_clean.xml. Keep the original raw extracts unchanged.

In [22]:
from pandas.api.types import is_numeric_dtype, is_integer_dtype, is_datetime64_any_dtype

amzn_raw = pd.read_csv("../data/raw/amzn_raw.csv")
products_raw = pd.read_csv("../data/raw/products_raw.csv")
book_raw = pd.read_csv("../data/raw/books_raw.csv")



# 2.1 Data profiling
# ----------------------------------------

# # Amazon Stocks:
print(amzn_raw.shape)  # row/columns counts
print(amzn_raw.dtypes)  # data types
print(amzn_raw.head()) # sample records
print(amzn_raw.isna().sum())   # missing values
print(amzn_raw.describe(include="all")) # ranges


# # Products:
print(products_raw.shape) # row/columns counts
print(products_raw.dtypes) # data types
print(products_raw.head()) # sample records
print(products_raw.isna().sum()) # missing values
print(products_raw.describe(include="all")) # ranges

# # Books:

print(book_raw.shape) # row/columns counts
print(book_raw.dtypes) # data types
print(book_raw.head()) # sample records
print(book_raw.isna().sum()) # missing values
print(book_raw.describe(include="all")) # ranges



# 2.2 Missing Values
# ----------------------------

amzn_clean = amzn_raw.copy()
book_clean = book_raw.copy()
products_clean = products_raw.copy()

# Check if there are missing values in the rows, I use boolean indexing

print(book_clean[book_clean.isna().any(axis=1)])

print(amzn_clean[amzn_clean.isna().any(axis=1)])

print(products_clean[products_clean.isna().any(axis=1)])

# We don't have to delete Missing values becuase they don't exist :)


# 2.2 Check duplicates
# ----------------------------

print(book_clean.duplicated(subset=["AbsoluteUrl"]).sum())
print(products_clean.duplicated(subset=["id"]).sum())
print(amzn_clean.duplicated(subset=["Date"]).sum())


# 2.3 Text and numbers in books:
# ----------------------------

book_clean["Price"] = pd.to_numeric(
        book_clean["Price"].str.replace("Â", "", regex=False).str.replace("£", "", regex=False),
        errors="coerce")

book_clean["Rating"] = book_clean["Rating"].map(rating_map)

# print(book_clean["Price"])



# 2.4 Use consistent snake_case headers
# ----------------------------

book_clean = book_clean.rename(columns={
    "Title": "title",
    "Price": "price",
    "Rating": "rating",
    "AbsoluteUrl": "absolute_url",
    "Page": "page"
})

amzn_clean.columns = amzn_clean.columns.str.lower().str.replace(" ", "_")
products_clean.columns = products_clean.columns.str.lower().str.replace(" ", "_").str.replace(".", "_")


#  validate price, rating, date, and stock OHLCV types

# validate books
print(is_numeric_dtype(book_clean["price"]))
print((book_clean["price"] >= 0).all())
print(is_numeric_dtype(book_clean["rating"]))
print((book_clean["rating"].between(1,5)).all())

# validate products
print(is_numeric_dtype(products_clean["price"]))
print(is_numeric_dtype(products_clean["rating_rate"]))
print(is_integer_dtype(products_clean["rating_count"]))


# validate amzn
amzn_clean["date"] = pd.to_datetime(amzn_clean["date"], utc=True)

print(is_datetime64_any_dtype(amzn_clean["date"])) # True

for col in ["open", "high", "low", "close"]:
        print(col, is_numeric_dtype(amzn_clean[col]))
        
print(is_integer_dtype(amzn_clean["volume"]))
print((amzn_clean["high"] >= amzn_clean["low"]).all())
print((amzn_clean["volume"] >= 0).all())


(251, 9)
Date                str
Open            float64
High            float64
Low             float64
Close           float64
Adj Close       float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object
                        Date        Open        High         Low       Close  \
0  2025-11-05 00:00:00-05:00  249.029999  251.000000  246.160004  250.199997   
1  2025-11-06 00:00:00-05:00  249.160004  250.380005  242.169998  243.039993   
2  2025-11-07 00:00:00-05:00  242.899994  244.899994  238.490005  244.410004   
3  2025-11-10 00:00:00-05:00  248.339996  251.750000  245.589996  248.399994   
4  2025-11-11 00:00:00-05:00  248.410004  249.750000  247.229996  249.100006   

    Adj Close    Volume  Dividends  Stock Splits  
0  250.199997  40610700        0.0           0.0  
1  243.039993  46004200        0.0           0.0  
2  244.410004  46374300        0.0           0.0  
3  248.399994  36476500        0.0           0.0  
4  249.100006  23564100  

### Save cleaned data

These transformations and exports were moved out of Phase 1. Book price and rating transformations are already in the preceding Phase 2 cell.


In [23]:
clean_path = Path("../data/clean")
clean_path.mkdir(parents=True, exist_ok=True)


# Books
book_clean.to_csv(clean_path / "books_clean.csv", index=False)



# Products
required_product_cols = ["id", "title", "price", "category", "rating_rate", "rating_count"]

products_clean = products_clean[required_product_cols]


with open(clean_path / "products_clean.json", "w", encoding="utf-8") as file:
    json.dump(products_clean.to_dict(orient="records"), file, indent=4, ensure_ascii=False)
   
    
products_clean.to_csv(clean_path / "products_clean.csv", index=False)


products_clean.to_xml(clean_path / "products_clean.xml")



# Amazon

required_cols = ["date", "open", "high", "low", "close", "volume"]
missing = [col for col in required_cols if col not in amzn_clean.columns]
if missing:
    print(f"Missing required AMZN columns: {missing}; clean file was not overwritten")
else:
    amzn_clean[required_cols].to_csv(clean_path / "amzn_clean.csv", index=False)
